# 08 · Synthesizing *verifiable* coding tasks

How to generate **new** coding problems and use a sandbox to
**verify** they're well-formed — the data-generation half of modern coding-agent
training (à la Qwen3-Coder-Next).

**Key concepts**
- **Verifiable task** — a problem paired with a reference solution *and* tests, where
  the reference actually passes the tests. Verification is *executable*, done by
  running it in a sandbox.
- **Discriminating tests** — good tests must also *fail* a trivial/wrong solution;
  the notebook checks this so tasks aren't accidentally trivial.
- **Executable environments as a filter** — the sandbox turns "the model made up a
  task" into "the task is provably valid," letting you build clean datasets at scale.

**How it works:** the LLM emits a task as JSON → the sandbox runs the reference
against the tests (must pass) and a stub (must fail) → keep only verified tasks → JSONL.

**Prerequisites:** AWS credentials + Bedrock model access.

In [ ]:
%pip install -q raitbox

### Configuration & generator
Bedrock client plus `generate_task`, which asks the model for a task as JSON (name / prompt / reference / tests).

In [ ]:
import json
import os
import re

import boto3

from raitbox import Sandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("ANTHROPIC_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

SYSTEM = [{"text": "You invent self-contained Python coding tasks. Reply with ONLY a "
           "JSON object with keys: name, prompt (what to implement), reference (a "
           "complete function definition), tests (>=3 assert statements calling it). "
           "No prose, no markdown fences."}]


def generate_task(topic: str) -> dict:
    resp = bedrock.converse(modelId=MODEL_ID, system=SYSTEM,
                            messages=[{"role": "user", "content": [{"text": f"Topic: {topic}"}]}],
                            inferenceConfig={"maxTokens": 1200, "temperature": 0.7})
    text = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(m.group(0))

### The verifier
Keep a task only if its reference solution passes its own tests **and** a trivial stub fails them (so the tests truly discriminate). `_as_code` tolerates the model returning code as a string or a list.

In [ ]:
def _as_code(v) -> str:
    # the model may return reference/tests as a string or a JSON list of lines
    return "\n".join(v) if isinstance(v, list) else str(v)


def is_verifiable(sbx, task) -> tuple[bool, str]:
    ref, tests = _as_code(task["reference"]), _as_code(task["tests"])
    # 1) reference solution must pass its own tests
    ex = sbx.run_code(ref + "\n" + tests + "\nprint('OK')")
    if ex.error is not None:
        return False, f"reference fails: {ex.error.name}: {ex.error.value}"
    if "OK" not in "".join(ex.logs.stdout):
        return False, "reference did not reach OK"
    # 2) tests must be discriminating: a trivial stub should FAIL them
    m = re.search(r"def\s+(\w+)", ref)
    if not m:
        return False, "no function definition in reference"
    stub = f"def {m.group(1)}(*a, **k):\n    return None\n" + tests + "\nprint('OK')"
    ex2 = sbx.run_code(stub)
    if ex2.error is None and "OK" in "".join(ex2.logs.stdout):
        return False, "tests pass even a trivial stub (not discriminating)"
    return True, "verified"

## Generate and validate a batch of tasks

In [ ]:
TOPICS = ["string manipulation", "arrays and hashing", "math", "recursion"]

verified = []
with Sandbox.create(region_name=REGION) as sbx:
    for topic in TOPICS:
        try:
            task = generate_task(topic)
        except Exception as e:
            print("gen failed:", e)
            continue
        ok, why = is_verifiable(sbx, task)
        print(f"{'KEEP' if ok else 'DROP'}  {task.get('name','?'):24} {why}")
        if ok:
            verified.append(task)

with open("verified_tasks.jsonl", "w") as f:
    for t in verified:
        f.write(json.dumps(t) + "\n")
print(f"\nverified {len(verified)}/{len(TOPICS)} -> verified_tasks.jsonl")

## Recap & where to go next
You generated candidate tasks and kept only the ones a sandbox could verify.

**Related:** `05`/`07` (consume tasks to score models) · `15` (train a policy on
synthesized, verified tasks).